# Spur App SDK — Design Spec

**Date:** 2026-06-10 · **Design epic:** `bd-bbig` · **Status:** approved design, pending implementation plan
**Depends on:** `2026-06-10-app-platform-contract-design.ipynb` (the contracts this SDK makes executable)

Open-source SDK for developing Spur notebook apps — by humans and, primarily, by agents. Principle: **the SDK is the contract made executable.** One tested implementation of each app⇄host seam, so no app ever reimplements a wire format from memory.

## 1. Problem & evidence

With a single gallery app shipped, the no-SDK tax is already measured:
- The notebook-MCP JSON-RPC client (`readExactly`/`readFrame`/`writeFrame`/`callNotebookTool`, ~60 lines) is duplicated **4× inside `app_gallery/html_video/app.ipynb`**.
- The Python server hand-parsed the port-store manifest and read paths that don't exist (`render.py:369`).
- `tests/test_packaging.py` hand-rolls a `.spurapp` packer simulating the Rust one.
- The bundled skill drifted to a phantom tool because nothing checks self-description against the live tool surface.

Every future app pays the same tax and risks the same bug classes.

## 2. Goals / non-goals

**Goals (v1 = full platform kit, per epic decision):**
1. `spur_app` Python package (PyPI) for app MCP servers.
2. `@spur/app` TypeScript package (JSR + npm) for Deno frontend cells.
3. Dev-loop tooling `spur app init|dev|doctor|pack|publish` — one Rust core, two front doors (notebook MCP tools for agents, standalone CLI for external developers).
4. `app-dev` agent skill shipped inside both packages and the repo.
5. Open-source publishing optimized for agentic retrieval: public mirror repo, PyPI/JSR/npm artifacts, `llms.txt`, JSON Schema for `spur-app.json`, examples. License: Apache-2.0.
6. html_video migrated onto the SDK as reference app and acceptance gate.

**Non-goals:**
- Host-side contract changes (companion spec).
- App store/registry distribution; paid distribution.
- Non-Python server runtimes for app MCP servers in v1 (manifest already supports `type`; SDK adds languages later).
- Replacing anywidget/AFM — the SDK wraps existing surfaces, it does not invent new widget machinery.

## 3. `spur_app` — Python server SDK (PyPI)

```python
from spur_app import App

app = App("html-video")          # FastMCP bootstrap, env contract, stderr logging

@app.tool()
def html_video_render(port_names: list[str], output_path: str, fps: int = 30):
    frames = [app.ports.read(p) for p in port_names]   # PortRead: bytes, mime, version, duration_sec
    out = app.artifacts.path(output_path)               # under SPUR_ARTIFACTS_DIR
    ...

if __name__ == "__main__":
    app.run()                     # stdio transport
```

**Modules:**
- `App` — wraps `mcp.server.fastmcp.FastMCP`; reads the provisioned env contract once at startup and fails fast with named-contract errors (e.g. `MissingCapabilityError("ports", "SPUR_PORTS_ROOT not provisioned — declare capabilities.ports in spur-app.json")`).
- `app.ports` — the port-store contract: parses `$SPUR_PORTS_ROOT/manifest.json`, reads `entry["path"]` (basename-joined under root), exposes `mime`, `version`, `duration_sec`. Pinned against the shared golden fixtures from the companion spec — the `@vN` path bug becomes impossible-by-construction.
- `app.artifacts` — paths under `SPUR_ARTIFACTS_DIR`.
- `app.env` — typed accessors for manifest-declared env (e.g. `TEMPLATES_DIR`).
- `spur_app.testing` — `FakePortStore` built from the same fixtures; pytest fixtures for tool invocation.

Pure-stdlib + `mcp` dependency only (keeps `uv run --with-requirements` fast).

## 4. `@spur/app` — Deno/TS frontend SDK (JSR + npm)

```ts
import { callTool, display, capture } from "@spur/app";

const result = await callTool("html_video_render", {
  port_names: ["spur-ad-capture"], output_path: "spur-ad.mp4", fps: 30,
});
display.html(`<video controls src="..." />`);
```

**Modules:**
- `callTool(name, args)` — the verified wire contract: connects to `Deno.env.get("SPUR_NOTEBOOK_MCP_SOCKET")` (injected per `src-tauri/commands.rs:1182-1190`), 4-byte big-endian length-framed JSON-RPC, `initialize` handshake, `tools/call`, structured-content unwrapping. Replaces the blob currently duplicated 4× per notebook — it is a port of working code, not a guess.
- `capture.canvas({ port, fps, durationSec, width, height })` — emits the `data-capture` contract consumed by `withVideoCapture` (`rendering.ts:9`): `<canvas data-capture="true" data-capture-cell-id=... data-capture-fps=... data-capture-duration-sec=...>`. Keeps the producer side of the capture loop in one place.
- `display.html/markdown/json` — `Symbol.for("Jupyter.display")` helpers.
- `ports` — typed helpers for `binds`/`emits` frontend-cell port bindings and `SPUR_NOTEBOOK_PORT_ROOT` reads.

Published to JSR (primary, Deno-native) with an npm mirror for tooling compatibility.

## 5. Dev-loop tooling — one Rust core, two front doors

**Core (Rust, in-daemon, next to `notebook_export_spur_app`/`notebook_import_spur_app`):**
- `init` — scaffold app dir: manifest (with `capabilities` + `skill`), `server/main.py` on `spur_app`, frontend cell templates on `@spur/app`, `skill/SKILL.md` from template, tests using `spur_app.testing`. Scaffolded app is doctor-green out of the box.
- `dev` — open the app against a live daemon with plugin hot-restart on server-file change.
- `doctor` — front door to `notebook_app_doctor` (companion spec §7).
- `pack` — the canonical `.spurapp` packer (existing `spur_app::archive`); app tests stop simulating it.
- `publish` — pack + checksums + doctor gate.

**Front door 1 — notebook MCP tools** (`notebook_app_init`, `notebook_app_doctor`, `notebook_app_pack`, …): agents are the primary app developers and already speak this surface.
**Front door 2 — standalone `spur app` CLI**: talks to a running daemon when available; otherwise invokes the same Rust core compiled into the CLI binary. **Binding constraint (epic decision): the PyPI/JSR packages never reimplement contract logic** — the CLI ships the Rust core, the SDKs ship only client-side contract readers pinned to fixtures.

## 6. Agent surface — the `app-dev` skill

Shipped at the repo root of the SDK, inside the PyPI sdist, and inside the JSR package. Documents the real loop:

1. Declare capabilities in `spur-app.json` (never read undeclared env).
2. `spur app init` / `notebook_app_init` to scaffold.
3. Server tools via `spur_app` (`app.ports.read`, `app.artifacts`); frontend via `@spur/app` (`callTool`, `capture.canvas`).
4. `doctor` green before commit; the HARD-GATE tool-name check keeps the app's own SKILL.md honest.
5. `pack`/`publish` through the canonical packer only.

The html_video skill rewrite (companion spec §8.4) becomes the first instance authored under this skill.

## 7. Open-source publishing & repo layout

**Source of truth: this monorepo**, `sdk/` directory — `sdk/python/` (spur_app), `sdk/typescript/` (@spur/app), `sdk/fixtures/` (symlink/copy-checked against `crates/spur-notebook/fixtures/port-store/`), `sdk/skill/`, `sdk/examples/`. Rationale (epic decision): conformance fixtures must move in lockstep with the Rust `PortStore` writer.

**Public mirror:** read-only repo (e.g. `github.com/<org>/spur-app-sdk`) synced from `sdk/` on release tags. Issues/PRs accepted on the mirror, applied to the monorepo.

**Published artifacts per release:** PyPI `spur-app`, JSR `@spur/app` (+npm mirror), `llms.txt` + docs designed for agent retrieval (stable URLs, one concept per page), **JSON Schema for `spur-app.json`** (generated from the Rust types — single source), runnable examples (html_video excerpt among them).

**License:** Apache-2.0. **Versioning:** SDK minor version tracks `spur.app` schema version; manifest gains optional `sdk_min`; doctor checks compatibility (this is what `jute_min` should have been). Fixture files carry a `contract_version`; SDKs refuse to read a newer major contract.

## 8. Task decomposition boundaries & acceptance

**DAG (each node ≈ one plan task; S-tasks depend on companion-spec tasks as noted):**
- U1 `sdk/` layout + fixture lockstep check — depends on companion T3 (fixtures exist).
- U2 `spur_app` Python SDK — depends U1; parallel with U3.
- U3 `@spur/app` TS SDK — depends U1 (fixtures for port reads); `callTool`/`capture`/`display` need nothing else and can start immediately.
- U4 tooling Rust core (`init|dev|pack|publish` + doctor front doors) — depends companion T1/T5; parallel with U2/U3.
- U5 standalone CLI binary — depends U4.
- U6 `app-dev` skill + docs/llms.txt/JSON Schema — depends U2–U4 APIs settling.
- U7 publishing pipeline (mirror sync, PyPI/JSR release) — depends U2, U3, U6.
- U8 html_video migration onto the SDK — depends U2, U3; acceptance gate.

**Acceptance criteria:**
- `spur app init` produces a doctor-green app; its tests pass via `spur_app.testing` with zero hand-written protocol code.
- html_video migrated: no hand-rolled socket client cells, `render.py` → `app.ports.read`, packaging test uses the canonical packer; full capture→render→MP4 e2e green.
- Fixture-lockstep CI: mutating the Rust port-store wire format without regenerating `sdk/fixtures` fails the build; same for the SDKs.
- Published `llms.txt` + JSON Schema retrievable at stable URLs; context7-style indexing verified against the mirror.
- **Success metric: a second gallery app costs ~1/10 of the first** (measured as lines of non-domain glue code).

## 9. References

- Companion spec: `2026-06-10-app-platform-contract-design.ipynb` (capabilities, provisioning, fixtures, doctor, trust).
- Epic `bd-bbig` decision trail: open-source + agent-retrieval optimization; v1 full kit; both front doors with single Rust core; two-spec decomposition; grounding verification (head `2e5d340a6`, `response_file_oids_match: true`).
- Evidence symbols: hand-rolled client ×4 (`app_gallery/html_video/app.ipynb`), `render.py:369`, `test_packaging.py` packer simulation, socket env injection (`src-tauri/commands.rs:51-52, 1182-1190`), capture contract (`rendering.ts:9-87`), packer (`spur_app.rs` + `archive`), lifecycle test (`mcp/mod.rs:5838`).
- Upstream inspiration: github.com/nexu-io/html-video (content-graph → frames → render pipeline; 21-template catalog model).